# BiLSTM vs DistilBERT: Model Comparison

This notebook compares the BiLSTM baseline classifier with a DistilBERT transformer
for philosophical question classification. The comparison covers accuracy, F1 score,
training time, and inference speed.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import pickle
import time
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Load Data

In [ ]:
DATA_DIR = Path.cwd().parent / 'data' / 'labels'
train_df = pd.read_csv(DATA_DIR / 'questions_train.csv')
test_df = pd.read_csv(DATA_DIR / 'questions_test.csv')

print(f"Train samples: {len(train_df)}")
print(f"Test samples: {len(test_df)}")
print(f"\nLabel distribution (train):")
print(train_df['label'].value_counts())
print(f"\nLabel distribution (test):")
print(test_df['label'].value_counts())

## 2. Load BiLSTM Model

In [ ]:
from src.classification.bilstm import BiLSTMClassifier, build_vocab

MODEL_DIR = Path.cwd().parent / 'models' / 'bilstm'

vocab_path = MODEL_DIR / 'vocab.pkl'
model_path = MODEL_DIR / 'final.pt'
label_path = MODEL_DIR / 'label2idx.json'

bilstm_clf = BiLSTMClassifier(vocab_size=1)
if model_path.exists():
    bilstm_clf.load(str(model_path), str(vocab_path), str(label_path))
    print("BiLSTM model loaded successfully")
else:
    print("BiLSTM model not found. Train first with: python -m src.classification.bilstm")

In [ ]:
# Evaluate BiLSTM on test set
if bilstm_clf.model is not None and vocab_path.exists():
    bilstm_preds = []
    bilstm_times = []
    
    for _, row in test_df.iterrows():
        start = time.time()
        label, conf, top3 = bilstm_clf.predict(row['question'])
        elapsed = time.time() - start
        bilstm_times.append(elapsed)
        bilstm_preds.append(label)
    
    bilstm_acc = accuracy_score(test_df['label'], bilstm_preds)
    bilstm_f1 = f1_score(test_df['label'], bilstm_preds, average='weighted')
    bilstm_avg_time = np.mean(bilstm_times)
    
    print(f"BiLSTM Accuracy: {bilstm_acc:.4f}")
    print(f"BiLSTM F1 (weighted): {bilstm_f1:.4f}")
    print(f"BiLSTM Avg inference: {bilstm_avg_time*1000:.2f} ms")
    print(f"\nClassification Report:")
    print(classification_report(test_df['label'], bilstm_preds))

## 3. Load/Train DistilBERT Model

In [ ]:
try:
    from src.classification.distilbert import DistilBERTClassifier
    DISTILBERT_AVAILABLE = True
except ImportError:
    DISTILBERT_AVAILABLE = False

print(f"DistilBERT available: {DISTILBERT_AVAILABLE}")

if DISTILBERT_AVAILABLE:
    distilbert_clf = DistilBERTClassifier()
    distilbert_model_path = Path.cwd().parent / 'models' / 'distilbert'
    
    if (distilbert_model_path / 'pytorch_model.bin').exists():
        distilbert_clf.load(str(distilbert_model_path / 'model.pt'))
        print("DistilBERT loaded from saved model")
    else:
        print("DistilBERT not fine-tuned. Using pre-trained only (zero-shot).")

In [ ]:
# Evaluate DistilBERT on test set
if DISTILBERT_AVAILABLE:
    distilbert_preds = []
    distilbert_times = []
    
    for _, row in test_df.iterrows():
        start = time.time()
        label, conf, top3 = distilbert_clf.predict(row['question'])
        elapsed = time.time() - start
        distilbert_times.append(elapsed)
        distilbert_preds.append(label)
    
    distilbert_acc = accuracy_score(test_df['label'], distilbert_preds)
    distilbert_f1 = f1_score(test_df['label'], distilbert_preds, average='weighted')
    distilbert_avg_time = np.mean(distilbert_times)
    
    print(f"DistilBERT Accuracy: {distilbert_acc:.4f}")
    print(f"DistilBERT F1 (weighted): {distilbert_f1:.4f}")
    print(f"DistilBERT Avg inference: {distilbert_avg_time*1000:.2f} ms")
    print(f"\nClassification Report:")
    print(classification_report(test_df['label'], distilbert_preds))

## 4. Comparison

In [ ]:
metrics = {
    'Model': ['BiLSTM', 'DistilBERT'],
    'Accuracy': [bilstm_acc, distilbert_acc] if DISTILBERT_AVAILABLE else [bilstm_acc, 0],
    'F1 Score': [bilstm_f1, distilbert_f1] if DISTILBERT_AVAILABLE else [bilstm_f1, 0],
    'Avg Inference (ms)': [bilstm_avg_time*1000, distilbert_avg_time*1000] if DISTILBERT_AVAILABLE else [bilstm_avg_time*1000, 0]
}
metrics_df = pd.DataFrame(metrics)
print(metrics_df.to_string(index=False))

In [ ]:
# Plot comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, metric in zip(axes, ['Accuracy', 'F1 Score', 'Avg Inference (ms)']):
    if DISTILBERT_AVAILABLE:
        ax.bar(['BiLSTM', 'DistilBERT'], [metrics[metric][0], metrics[metric][1]],
               color=['#4C72B0', '#DD8452'])
        for i, v in enumerate([metrics[metric][0], metrics[metric][1]]):
            ax.text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=11)
    else:
        ax.bar(['BiLSTM'], [metrics[metric][0]], color=['#4C72B0'])
        ax.text(0, metrics[metric][0] + 0.01, f'{metrics[metric][0]:.3f}', ha='center', fontsize=11)
    ax.set_title(metric, fontsize=13)
    ax.set_ylim(0, max(metrics[metric]) * 1.3 if max(metrics[metric]) > 0 else 1)

plt.tight_layout()
plt.savefig(Path.cwd().parent / 'notebooks' / 'model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Confusion Matrices

In [ ]:
if DISTILBERT_AVAILABLE:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    for ax, (name, preds) in enumerate(zip(
        ['BiLSTM', 'DistilBERT'],
        [bilstm_preds, distilbert_preds]
    )):
        cm = confusion_matrix(test_df['label'], preds)
        labels = sorted(test_df['label'].unique())
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[ax],
                    xticklabels=labels, yticklabels=labels)
        axes[ax].set_title(f'{name} Confusion Matrix')
        axes[ax].set_xlabel('Predicted')
        axes[ax].set_ylabel('True')
    
    plt.tight_layout()
    plt.show()
else:
    cm = confusion_matrix(test_df['label'], bilstm_preds)
    labels = sorted(test_df['label'].unique())
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=labels, yticklabels=labels)
    plt.title('BiLSTM Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.show()

## 6. Conclusions

| Metric | BiLSTM | DistilBERT |
|--------|--------|------------|
| Parameters | ~200K | ~67M |
| Training time | ~5 min (CPU) | ~30 min (GPU recommended) |
| Inference | Fast (CPU) | Slower (GPU recommended) |
| Fine-tuning needed | Yes | Yes (pre-trained) |
| Best for | Real-time, resource-constrained | Higher accuracy with GPU |

**BiLSTM** is a lightweight baseline suitable for real-time classification on CPU.
**DistilBERT** offers transformer-level accuracy but requires GPU for practical training.